# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I chose Logistic Regression because the goal is to predict whether a page is declining or not.

I wanted to start with a simple model that is easy to understand and explain. It also gives us a score that we can use to rank the pages and decide which ones should be reviewed first.

I will compare the model with my Week-4 baseline using the same data and evaluation metrics. This will help me see if the model actually improves on the simple rule I created in Week 4.


In [26]:
# Get the FlyRank project files

!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 299, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 299 (delta 130), reused 98 (delta 98), pack-reused 106 (from 1)
Receiving objects: 100% (299/299), 1.88 MiB | 4.91 MiB/s, done.
Resolving deltas: 100% (161/161), done.


In [27]:
# Move into the FlyRank project folder

%cd flyrank-ml-internship-starter

!ls

/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [28]:
# Check that the raw dataset is available

!ls data/raw

content_refresh_anonymized.csv


In [29]:
# Prepare the features for the Week 5 model

!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv


In [30]:
# Check that the processed feature file was created

!ls data/processed

feature_metadata.json  refresh_feature_vector.csv


In [31]:
# Load the prepared feature data

import pandas as pd

feature_path = "data/processed/refresh_feature_vector.csv"

features = pd.read_csv(feature_path)

print("Number of rows:", len(features))
print("Number of columns:", len(features.columns))

print("\nTarget values:")
print(features["is_declining_label"].value_counts())

print("\nDeclining rate:")
print(round(features["is_declining_label"].mean(), 3))

Number of rows: 30000
Number of columns: 52

Target values:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining rate:
0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I will split the data by client instead of randomly splitting the pages.

I chose this because pages from the same client can be similar. If the same client appears in both the training and testing data, the results might look better than they really are.

So I will use about 80% of the clients for training and the other 20% for testing. This gives me a more realistic idea of how the model performs on clients it has not seen before.


In [33]:
# Split the data by client

import numpy as np

groups = features["client_id"].fillna("unknown").astype(str)

unique_clients = groups.drop_duplicates().to_numpy()

random_generator = np.random.default_rng(42)

shuffled_clients = random_generator.permutation(unique_clients)

test_client_count = max(
    1,
    int(round(len(shuffled_clients) * 0.20))
)

test_clients = set(shuffled_clients[:test_client_count])

test_mask = groups.isin(test_clients)

train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

X = features.drop(columns=["is_declining_label"])
y = features["is_declining_label"]

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training clients:", groups.iloc[train_idx].nunique())
print("Testing clients:", groups.iloc[test_idx].nunique())

Training rows: 27675
Testing rows: 2325
Training clients: 26
Testing clients: 6


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [34]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


I will train the Logistic Regression model using the training data and then test it on the clients that were held out.

I will compare its results with my Week-4 baseline using the same test data. The main thing I want to see is whether the model is better at putting declining pages near the top of the list.


In [35]:
# Prepare the data for Logistic Regression

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

drop_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

X = features.drop(columns=drop_columns)
y = features["is_declining_label"]

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numeric_columns = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_columns
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_columns
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "logistic_regression",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [36]:
# Evaluate the Logistic Regression model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1:", round(f1_score(y_test, y_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("Average Precision:", round(average_precision_score(y_test, y_prob), 3))

Accuracy: 0.791
Precision: 0.823
Recall: 0.592
F1: 0.688
ROC AUC: 0.864
Average Precision: 0.835


In [37]:
# Check how well the model ranks declining pages

test_results = pd.DataFrame({
    "actual": y_test.values,
    "score": y_prob
})

test_results = test_results.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

for k in [20, 50, 100]:
    precision_at_k = test_results.head(k)["actual"].mean()
    print(f"Precision@{k}:", round(precision_at_k, 3))

Precision@20: 1.0
Precision@50: 1.0
Precision@100: 1.0


In [38]:
# Prepare the features for the model

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

model_numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

model_categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

X = features[
    model_numeric_features + model_categorical_features
].copy()

y = features["is_declining_label"]

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            model_numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            model_categorical_features
        )
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "logistic_regression",
        LogisticRegression(
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
    )
])

model.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")
print("Number of numeric features:", len(model_numeric_features))
print("Number of categorical features:", len(model_categorical_features))

Logistic Regression model trained successfully.
Number of numeric features: 18
Number of categorical features: 8


In [39]:
# Evaluate the corrected Logistic Regression model

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1:", round(f1_score(y_test, y_pred), 3))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("Average Precision:", round(average_precision_score(y_test, y_prob), 3))

Accuracy: 0.663
Precision: 0.57
Recall: 0.559
F1: 0.564
ROC AUC: 0.704
Average Precision: 0.525


In [40]:
# Check how well the corrected model ranks declining pages

test_results = pd.DataFrame({
    "actual": y_test.values,
    "score": y_prob
})

test_results = test_results.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

for k in [20, 50, 100]:
    precision_at_k = test_results.head(k)["actual"].mean()
    print(f"Precision@{k}:", round(precision_at_k, 3))

Precision@20: 0.35
Precision@50: 0.4
Precision@100: 0.43


In [41]:
# Compare the model with the Week-4 baseline

baseline_precision_at_50 = 0.24
model_precision_at_50 = 0.40

improvement = model_precision_at_50 - baseline_precision_at_50

print("Week-4 baseline Precision@50:", baseline_precision_at_50)
print("Logistic Regression Precision@50:", model_precision_at_50)
print("Improvement:", round(improvement, 2))

Week-4 baseline Precision@50: 0.24
Logistic Regression Precision@50: 0.4
Improvement: 0.16


The Logistic Regression model did better than my Week-4 baseline.

The baseline had a Precision@50 of 0.24, while the Logistic Regression model got 0.40. That is an improvement of 0.16.

The model also got 0.35 at Precision@20 and 0.43 at Precision@100. Overall, the model was better at putting declining pages near the top of the list.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [42]:
# Look at the pages the model got wrong

errors = X_test.copy()

errors["actual"] = y_test.values
errors["predicted"] = y_pred
errors["score"] = y_prob

wrong_predictions = errors[
    errors["actual"] != errors["predicted"]
].copy()

print("Number of wrong predictions:", len(wrong_predictions))

print("\nFalse positives:")
print(
    wrong_predictions[
        (wrong_predictions["actual"] == 0) &
        (wrong_predictions["predicted"] == 1)
    ][["actual", "predicted", "score"]].head(3)
)

print("\nFalse negatives:")
print(
    wrong_predictions[
        (wrong_predictions["actual"] == 1) &
        (wrong_predictions["predicted"] == 0)
    ][["actual", "predicted", "score"]].head(3)
)

Number of wrong predictions: 784

False positives:
     actual  predicted     score
168       0          1  0.703063
198       0          1  0.826032
251       0          1  0.549078

False negatives:
     actual  predicted     score
48        1          0  0.474704
141       1          0  0.422207
275       1          0  0.477889


In [43]:
# Add the page IDs so we can inspect a few wrong predictions

error_details = features.iloc[test_idx][
    ["content_id", "client_id"]
].copy()

error_details["actual"] = y_test.values
error_details["predicted"] = y_pred
error_details["score"] = y_prob

false_positives = error_details[
    (error_details["actual"] == 0) &
    (error_details["predicted"] == 1)
].copy()

false_negatives = error_details[
    (error_details["actual"] == 1) &
    (error_details["predicted"] == 0)
].copy()

print("Example false positives:")
print(false_positives.head(3))

print("\nExample false negatives:")
print(false_negatives.head(3))

Example false positives:
               content_id          client_id  actual  predicted     score
168  content_d7cbd76b788d  client_f74efabef1       0          1  0.703063
198  content_c3e86d4031b6  client_f74efabef1       0          1  0.826032
251  content_7dff534db3ae  client_f74efabef1       0          1  0.549078

Example false negatives:
               content_id          client_id  actual  predicted     score
48   content_326fa2fa449f  client_98a3ab7c34       1          0  0.474704
141  content_0af426466565  client_f74efabef1       1          0  0.422207
275  content_ea851c8c0ad2  client_f74efabef1       1          0  0.477889


In [44]:
# Inspect the features of a few wrong predictions

wrong_ids = list(false_positives["content_id"].head(3)) + \
            list(false_negatives["content_id"].head(3))

wrong_cases = features[
    features["content_id"].isin(wrong_ids)
].copy()

columns_to_show = [
    "content_id",
    "content_type",
    "main_intent",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "impressions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "is_declining_label"
]

print(wrong_cases[columns_to_show].to_string(index=False))

          content_id    content_type   main_intent  content_age_days  impressions_90d  clicks_90d  sessions_90d  impressions_last_30d  impressions_prev_30d  ctr  avg_position  engagement_rate  is_declining_label
content_326fa2fa449f keyword article informational                91                4           0             2                     0                     3 0.00           8.3             0.00                   1
content_0af426466565 keyword article informational                91                9           0             2                     0                     8 0.00           3.6             0.00                   1
content_d7cbd76b788d keyword article    commercial               144            17992          19            36                  6024                  6854 0.11           6.4             0.00                   0
content_c3e86d4031b6 keyword article transactional               175              801           0            11                   801                   

The model made 784 wrong predictions on the test data.

I looked at a few of the wrong predictions to understand what went wrong. Some of the pages the model missed had very little traffic. For example, two of the pages had only 4 and 9 impressions in 90 days. Because there was not much data, it was harder for the model to identify them as declining.

For some of the false positives, the pages had more traffic, but the model still predicted that they were declining. This shows that the model can sometimes get confused when the recent performance of a page changes.

Overall, the model is useful for finding pages that may need attention, but it will not get every page right. Pages with very little traffic seem to be harder for the model to predict.


In [45]:
# Check which features have the biggest effect on the model

feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["logistic_regression"].coef_[0]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "absolute_coefficient": abs(coefficients)
})

feature_importance = feature_importance.sort_values(
    "absolute_coefficient",
    ascending=False
)

print(feature_importance.head(10)[
    ["feature", "coefficient"]
].to_string(index=False))

                                 feature  coefficient
            numeric__log_impressions_90d     1.644726
                     numeric__word_count     1.613842
                     numeric__char_count    -1.357131
  categorical__impression_tier_excellent    -0.743246
        categorical__position_tier_top_3    -0.729542
        categorical__impression_tier_low     0.662666
                 numeric__log_clicks_90d    -0.652299
categorical__content_type_feedly article    -0.469449
  categorical__word_count_tier_2000-3500    -0.460014
        categorical__main_intent_unknown     0.448980


###Interpetation

The model relied most on things like impressions, word count, character count, clicks, and search position.

The strongest positive coefficient was `log_impressions_90d`, while `word_count` was also important. Some features had negative coefficients, such as `char_count` and `log_clicks_90d`.

This gives me an idea of which parts of the content and its performance are having the most influence on the model's predictions. However, these results show relationships in the model and do not mean that one feature directly causes a page to decline.


In [46]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self-check

* I completed all four sections.
* I used a client-level split for the training and testing data.
* I made sure there was no data leakage.
* I did not use `trend_direction` or `trend_pct` as features.
* I trained the Logistic Regression model successfully.
* I compared the model with my Week-4 baseline.
* I checked Precision@20, Precision@50, and Precision@100.
* I looked at some of the predictions the model got wrong.
* I checked which features had the biggest influence on the model.
* I explained the results without overstating what the model can do.
* The notebook runs without errors.
* The notebook is saved in `work/notebooks/w05_model.ipynb`.
